## Tutorial 1 - Preparing a protein with Martini 3

There are a few different methods you can use to set up Martini membrane protein simulations, such as the [Martini Maker](https://www.charmm-gui.org/?doc=input/martini.bilayer) on Charmm-GUI. We will use a more flexible setup below, so you can explore all options and could potentially use the sets to automate future system assemblies if you want to do this in a high-throughput way. 

First step will be to convert our atomistic representation of a protein into a coarse-grained (CG) Martini version, where atoms are replaced by beads representing that residue. To get an atomistic representation, you can get a prediction from something such as AlphaFold or use an experimentally resolved structure. Please bear in mind that the CG representation _will only be as good as your atomistic input_, so it is well worth taking time to make sure this is as accurate as possible. Possible considerations for your atomistic structure are:

<details>
<summary>    
Are there any missing loops/residues?
</summary>

It is always worth fixing missing loops if possible, and this can be done in several ways. [SwissModel](https://swissmodel.expasy.org/) is a great online server to help with this. [Modeller](https://salilab.org/modeller/) is a locally installable tool which is great, especially for flexible loops. The predicted AlphaFold structure might also be of use to complete certain structures.

</details>

<details>
<summary>    
Protonation state of residues
</summary>

This can be an important consideration, and there are online tools to help with assignments of protonation states such as the [H++ server](http://newbiophysics.cs.vt.edu/H++/). The [propka tool](https://propka.readthedocs.io/en/latest/) can run on the command line for static proteins as well.

</details>

<details>
<summary>    
Are there any partner proteins?
</summary>

This very much depends on the questions you are asking, but well worth considering. If there are no experimental structures, structure prediction tools (such as AlphaFold) can be used to generate starting structures.  

</details> 

We also need to orient our protein with respect to the membrane. There are a few different tools you can use to do this, but we are going to use the [PPM webserver](https://opm.phar.umich.edu/ppm_server3_cgopm) to get our orientation. You should be able to find our protein structure that has been fixed (see above) in `tutorial_1` that you can use.  

The protein we are going to simulate today is VDAC1, in this case from a mouse (original PDB file can be found [here](https://www.rcsb.org/structure/3EMN)). On the server, most membrane proteins that are found in the PDB have already been oriented, with the example of [our protein of interest today](https://opm.phar.umich.edu/proteins/836).   

But, in the interest of showing what you can do, we can also use the server to set it up. VDAC1 is a voltage-dependent anion channel found in the outer membrane of mitochondria (it's a pretty cool protein if you are interested). The N-terminus of the protein is facing downwards (if we think in the z-direction) into the mitochondrial intermembrane space. We can use these bits of knowledge to set up the PPM server.  

Feel free to try this yourself, but the server might get busy. The output is found at `tutorial_1/VDAC1_ppm.pdb`   

![PPM3.0 server options](images/PPM_inputs.png)

We can now visulize our protein! You can bypass most of the below cell, it is just configuring visulization in the notebook. Press run and an interactive viewer should appear below:

In [ ]:
from IPython.display import HTML, display
import base64, os


def view_atomistic(
    structure_file: str,
    height: int = 600,
    show_cartoon: bool = True,        # Ribbon/cartoon for secondary structure
    show_ball_and_stick: bool = True, # Bonds + atoms for all heavy atoms
    show_spacefill: bool = False,     # VdW spheres (CPU-heavy for large systems)
    cartoon_alpha: float = 0.85,      # Cartoon opacity
    stick_size: float = 0.16,         # Bond stick radius in Å
    ball_size: float = 0.25,          # Atom sphere radius factor (ball-and-stick)
    spacefill_alpha: float = 0.25,    # VdW sphere opacity (if enabled)
) -> HTML:
    """
    Visualise an atomistic protein structure in Mol* inside Jupyter.

    Representations
    ---------------
    cartoon       → ribbon coloured by secondary structure (helix/sheet/coil)
    ball-and-stick→ heavy atom spheres + covalent bonds as sticks
    spacefill     → full VdW spheres (optional; disabled by default for speed)

    Parameters
    ----------
    structure_file : str
        Path to PDB, mmCIF/CIF, or GRO file.

    height : int
        Viewer height in pixels.

    show_cartoon : bool
        Render cartoon/ribbon secondary-structure representation.

    show_ball_and_stick : bool
        Render bonds and atoms as balls and sticks.

    show_spacefill : bool
        Overlay semi-transparent VdW spheres.  Good for surface visualisation
        but expensive for large systems (>50 k atoms).

    cartoon_alpha : float
        Cartoon opacity (0–1).

    stick_size : float
        Bond stick radius in Å.

    ball_size : float
        Atom sphere scale factor for ball-and-stick (relative to element radius).

    spacefill_alpha : float
        Opacity of VdW spheres (0–1).  Only used when show_spacefill=True.

    Returns
    -------
    IPython.display.HTML
        Call display() on the return value, or let Jupyter auto-display it.
    """
    # ── Load and encode structure file ────────────────────────────────────────
    with open(structure_file, "rb") as fh:
        b64 = base64.b64encode(fh.read()).decode()

    ext = os.path.splitext(structure_file)[1].lstrip(".").lower()
    fmt = {"gro": "gro", "pdb": "pdb", "cif": "mmcif", "mmcif": "mmcif"}.get(ext, "pdb")

    # Convert Python bools to JS booleans
    js_cartoon        = str(show_cartoon).lower()
    js_ball_and_stick = str(show_ball_and_stick).lower()
    js_spacefill      = str(show_spacefill).lower()

    page = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8"/>
  <link rel="stylesheet"
        href="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.css"/>
  <style>
    html, body {{ margin:0; padding:0; height:100%; background:#1a1a2e; }}
    #app {{ position:absolute; inset:0; }}
  </style>
</head>
<body>
  <div id="app"></div>
  <script src="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.js"></script>
  <script>
  (async () => {{

    // ── 1. Create Mol* viewer ────────────────────────────────────────────────
    const viewer = await molstar.Viewer.create('app', {{
      layoutIsExpanded:       false,
      layoutShowControls:     false,
      layoutShowLeftPanel:    false,
      layoutShowSequence:     false,   // sequence bar useful for atomistic work
      layoutShowLog:          false,
      viewportShowAnimation:  false,
      viewportShowExpand:     true,
    }});
    const plugin = viewer.plugin;

    // ── 2. Decode base64 structure ───────────────────────────────────────────
    const structText = atob("{b64}");

    // ── 3. Build the structure ───────────────────────────────────────────────
    const rawData   = await plugin.builders.data.rawData(
      {{ data: structText }},
      {{ state: {{ isGhost: true }} }}
    );
    const trajectory = await plugin.builders.structure.parseTrajectory(rawData, '{fmt}');
    const model      = await plugin.builders.structure.createModel(trajectory);
    const structure  = await plugin.builders.structure.createStructure(model);

    const colorTheme = {{ name: 'chain-id' }};

    // ── 4. Cartoon / ribbon ──────────────────────────────────────────────────
    if ({js_cartoon}) {{
      await plugin.builders.structure.representation.addRepresentation(structure, {{
        type: 'cartoon',
        colorTheme: colorTheme,
        typeParams: {{
          alpha: {cartoon_alpha},
        }},
      }});
    }}

    // ── 5. Ball-and-stick ────────────────────────────────────────────────────
    if ({js_ball_and_stick}) {{
      await plugin.builders.structure.representation.addRepresentation(structure, {{
        type: 'ball-and-stick',
        colorTheme: colorTheme,
        typeParams: {{
          sizeFactor:     {stick_size},
          ballSizeFactor: {ball_size},
          bondSpacing:    1.0,
        }},
      }});
    }}

    // ── 6. VdW spacefill (optional) ──────────────────────────────────────────
    if ({js_spacefill}) {{
      await plugin.builders.structure.representation.addRepresentation(structure, {{
        type: 'spacefill',
        colorTheme: colorTheme,
        sizeTheme:  {{ name: 'physical' }},   // true VdW radii per element
        typeParams: {{
          alpha: {spacefill_alpha},
        }},
      }});
    }}

    // ── 7. Fit camera ────────────────────────────────────────────────────────
    plugin.canvas3d?.requestCameraReset();

  }})();
  </script>
</body>
</html>"""

    escaped = page.replace("'", "&#39;")
    return HTML(
        f'<iframe srcdoc=\'{escaped}\' '
        f'style="width:100%;height:{height}px;border:none;border-radius:6px;"></iframe>'
    )


### Below is the lines that actually executes the visulization

display(view_atomistic("VDAC1_ppm.pdb", show_ball_and_stick=True))


You can see the dummy beads added by the PPM server in blue and red, while the protein is in green. Hopefully you can see that the protein spans the (predicted) membrane.

While the dummy beads (DUM) are great for showing where the membrane should be, we need to remove them before our next steps. We can do that using a simple command:

In [ ]:
!grep -v DU VDAC1_ppm.pdb > VDAC1_clean.pdb

We are now ready to convert our protein to a Martini 3 representation. Let's have a look at the possible options when using [martinize](https://elifesciences.org/reviewed-preprints/90627), which is the tool we are going to use to convert between AT and CG resolutions: 

In [ ]:
!martinize2 -h

There are many options for the input here, but we will focus on the ones used in this tutorial:

- `-f` which specifies the input file for your protein. Need to make sure you have removed anything that isn't protein. This will be either a .pdb or .gro file
- `-x` the output coordinate file for the Martini coordinate file. This will usually be a .pdb file
- `-o` the topology file that will be created; this points towards any .itp files that are created. This will be a .top file
- `-name` this sets the name of the protein, and is not strictly needed. This becomes useful when simulating multi-protein systems
- `-dssp` the flag used to determine structural features, such as alpha helcies/beta sheets etc., so the correct parameters can be used. An executable pointing towards a way to run dssp can be specified after this flag; otherwise, mdtraj will be used
- `-elastic` this flag will ensure elastic bonds are written. Without this (or the use of GoMartini), there will be no secondary structure retention and the protein will unfold without the use of other restraints
- `-ef` the force constant used for the elastic network (i.e. how strong the elastic bands are) in kJ mol<sup>-1</sup> nm<sup>-2</sup>. The default value is 700 kJ mol<sup>-1</sup> nm<sup>-2</sup>
- `-el` the lower boundary for elastic networks (in nm). The default value is 0 nm
- `-eu` the upper boundary for elastic networks (in nm). The default value is 0.9 nm


The default values are normally a good place to start for the flags that specify a value. See if you can put together a martinize2 command to convert our protein from AT to CG resolution:

In [ ]:
!martinize2 -f VDAC1_clean.pdb -x VDAC1_cg.pdb -o topol.top -name VDAC1 -dssp -elastic -ef 700 -el 0 -eu 0.8

<details>
<summary>    
<i>Really</i> stuck? Click on this to reveal a command we can use
</summary>

`!martinize2 -f VDAC1_clean.pdb -x VDAC1_cg.pdb -o topol.top -name VDAC1 -dssp -elastic -ef 700 -el 0 -eu 0.8`  

If there's anything you don't understand, please ask!

</details>

Other commands that might be useful with martinize with your own systems:

- `-merge` if you have multiple protein chains (in a complex, for example) this treats them as one for coarse-graining, and will make elastic bonds between the different chains to retain tertiary structure
- `-id-regions` which applies the new [Martini3-IDP forcefield](https://www.nature.com/articles/s41467-025-58199-2) to the specified regions so they have behaviour that resembles intrinsically disordered (ID) proteins. This ensures there are no elastic bonds in this region
- `-go` (and related flags) will construct a [GoMartini](https://www.nature.com/articles/s41467-025-58719-0) network, which would be used _instead_ of an elastic network to retain secondary structure. This method is slightly more computationally expensive, but can capture conformational rearrangements
- `-modify` this lets us make changes to the protein, such as mutations, but also alter the protonation state of any important residues.


Now let's visualise our protein!

In [ ]:
def view_martini(
    structure_file: str,
    height: int = 550,
    spacefill_size: float = 2.6,   # Å — Regular bead radius (Martini 3)
    sphere_alpha: float = 0.2,    # 0 = transparent, 1 = fully opaque
    stick_radius: float = 0.3,    # Å — bond stick radius
) -> HTML:
    """
    Visualise a Martini 3 coarse-grained protein in Mol* inside Jupyter.

    CG beads  →  semi-transparent VdW spacefill spheres
    Bonds     →  sticks (inferred from bead distances, or from CONECT records)

    Parameters
    ----------
    structure_file : str
        Path to your CG structure. PDB is recommended; GRO is also supported.
        For accurate bond connectivity, generate a PDB with CONECT records
        from your ITP (e.g. via gmx editconf or a conversion script).
        Without CONECT records Mol* infers bonds from bead-bead distances —
        this works well for backbone beads (~3.8 Å) but may miss or add
        spurious sidechain bonds.

    height : int
        Height of the embedded viewer in pixels (default 550).

    sphere_alpha : float
        Sphere opacity.  Semi-transparent (< 1) lets you see bonds inside.

    stick_radius : float
        Bond stick radius in Å.

    Returns
    -------
    IPython.display.HTML
        Call display() on the return value, or let Jupyter auto-display it.
    """
    # ── Load and encode structure file ────────────────────────────────────────
    with open(structure_file, "rb") as fh:
        b64 = base64.b64encode(fh.read()).decode()

    ext = os.path.splitext(structure_file)[1].lstrip(".").lower()
    fmt = {"gro": "gro", "pdb": "pdb", "cif": "mmcif", "mmcif": "mmcif"}.get(ext, "pdb")

    # ── Build the embedded HTML page ──────────────────────────────────────────
    # Note: Python f-string — all JS curly braces must be doubled {{ }}
    #       except the variables being interpolated: {b64}, {fmt}, etc.
    page = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8"/>
  <link rel="stylesheet"
        href="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.css"/>
  <style>
    html, body {{ margin:0; padding:0; height:100%; background:#1a1a2e; }}
    #app {{ position:absolute; inset:0; }}
  </style>
</head>
<body>
  <div id="app"></div>
  <script src="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.js"></script>
  <script>
  (async () => {{

    // ── 1. Create Mol* viewer ────────────────────────────────────────────────
    const viewer = await molstar.Viewer.create('app', {{
      layoutIsExpanded:       false,
      layoutShowControls:     false,
      layoutShowLeftPanel:    false,
      layoutShowSequence:     false,
      layoutShowLog:          false,
      viewportShowAnimation:  false,
      viewportShowExpand:     true,
    }});
    const plugin = viewer.plugin;

    // ── 2. Decode base64 structure (PDB / GRO are plain ASCII) ──────────────
    const structText = atob("{b64}");

    // ── 3. Build the structure using the plugin builder API ─────────────────
    // rawData → parseTrajectory → createModel → createStructure
    const rawData = await plugin.builders.data.rawData(
      {{ data: structText }},
      {{ state: {{ isGhost: true }} }}       // hide raw data node from the UI tree
    );

    const trajectory = await plugin.builders.structure.parseTrajectory(rawData, '{fmt}');
    const model      = await plugin.builders.structure.createModel(trajectory);
    const structure  = await plugin.builders.structure.createStructure(model);

    // ── 4. VdW spacefill — one sphere per CG bead ───────────────────────────
    await plugin.builders.structure.representation.addRepresentation(structure, {{
      type:  'spacefill',
      colorTheme: {{ name: 'chain-id' }},
      sizeTheme:  {{
        name:   'uniform',
        params: {{ value: {spacefill_size} }},   // explicit radius in Å
      }},
      typeParams: {{
        alpha: {sphere_alpha},                   // semi-transparent: bonds show through
      }},
    }});

    // ── 5. Bond sticks ───────────────────────────────────────────────────────
    //   Mol* automatically infers bonds from bead-bead distances.
    //   Supply CONECT records in your PDB for exact Martini connectivity.
    await plugin.builders.structure.representation.addRepresentation(structure, {{
      type:  'ball-and-stick',
      colorTheme: {{ name: 'chain-id' }},
      sizeTheme:  {{ name: 'uniform' }},
      typeParams: {{
        sizeFactor:     {stick_radius},
        ballSizeFactor: 0.01,   // near-zero ball; spheres handled by spacefill above
        bondSpacing:    1.0,
      }},
    }});

    // ── 6. Fit camera to the loaded structure ────────────────────────────────
    plugin.canvas3d?.requestCameraReset();

  }})();
  </script>
</body>
</html>"""

    # Escape single quotes so the string is safe as an HTML attribute value
    escaped = page.replace("'", "&#39;")
    return HTML(
        f'<iframe srcdoc=\'{escaped}\' '
        f'style="width:100%;height:{height}px;border:none;border-radius:6px;"></iframe>'
    )

#### Below here is the line to change

display(view_martini("VDAC1_cg.pdb", spacefill_size=2.6, sphere_alpha=0.2, stick_radius=0.3))


<details>
<summary>    
<b>Question: how would you describe the changes between the atomistic structure earlier in the notebook and this CG structure?</b>
</summary>

There are many less atoms in the CG structure than in the atomistic structure, as the atoms have been grouped together into particles. You might notice that many of the side chains are ony represented by 1 bead and it is difficult to tell them apart visually. However, they will have different particle <I>types</I> so will behave differently.

</details>

<details>
<summary>    
<b><I>Hard</I> question: how do you think you could evaluate if you are using a good force constant value?</b>
</summary>

One method is to compare the root mean squared fluctuation (RMSF) of an atomistic simulation of your protein with a CG simulation. If regions are too flexible/rigid you can then change the force constant and the cutoffs used! This depends on how important capturing the flexibility of the protein is to the question you are asking, the default values <i>do a pretty good job</i> for many systems.

</details>